## Import libraries

In [ ]:
from pathlib import Path

import json
import pickle

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

In [2]:
df = pd.read_excel("../data/Telco_customer_churn.xlsx")

df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
df.dropna(inplace=True)


In [3]:
print(df.head())
print(df.info())

   CustomerID  Count        Country       State         City  Zip Code  \
0  3668-QPYBK      1  United States  California  Los Angeles     90003   
1  9237-HQITU      1  United States  California  Los Angeles     90005   
2  9305-CDSKC      1  United States  California  Los Angeles     90006   
3  7892-POOKP      1  United States  California  Los Angeles     90010   
4  0280-XJGEX      1  United States  California  Los Angeles     90015   

                 Lat Long   Latitude   Longitude  Gender  ...        Contract  \
0  33.964131, -118.272783  33.964131 -118.272783    Male  ...  Month-to-month   
1   34.059281, -118.30742  34.059281 -118.307420  Female  ...  Month-to-month   
2  34.048013, -118.293953  34.048013 -118.293953  Female  ...  Month-to-month   
3  34.062125, -118.315709  34.062125 -118.315709  Female  ...  Month-to-month   
4  34.039224, -118.266293  34.039224 -118.266293    Male  ...  Month-to-month   

  Paperless Billing             Payment Method  Monthly Charges Tota

In [ ]:
base_dir = Path.cwd()
if not (base_dir / "data").exists():
    base_dir = base_dir.parent

data_path = base_dir / "data" / "Telco_customer_churn.xlsx"
artifact_dir = base_dir / "app" / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / "churn_model.pkl"
metadata_path = artifact_dir / "churn_model_meta.json"

feature_cols = [
    "Gender",
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Tenure Months",
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method",
    "Monthly Charges",
    "Total Charges",
    "CLTV",
]
num_cols = ["Tenure Months", "Monthly Charges", "Total Charges", "CLTV"]
cat_cols = [feature for feature in feature_cols if feature not in num_cols]

df = pd.read_excel(data_path)
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
df["CLTV"] = pd.to_numeric(df["CLTV"], errors="coerce")
df.dropna(subset=["Total Charges", "CLTV", "Churn Value"], inplace=True)

X = df[feature_cols].copy()
y = df["Churn Value"].astype(int).copy()

print(X.head())
print(X.info())

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

def _to_string_frame(values):
    return values.astype(str)

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("to_str", FunctionTransformer(_to_string_frame)),
    ("onehot", ohe),
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols),
])

model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = "roc_auc" if y.nunique() > 1 else "accuracy"
scores = cross_val_score(model, X, y, cv=cv, scoring=scoring, error_score="raise")

model.fit(X, y)

with model_path.open("wb") as f:
    pickle.dump(model, f)

metadata = {
    "target": "Churn Value",
    "features": [
        {
            "name": col,
            "kind": "numeric" if col in num_cols else "categorical",
            **(
                {
                    "min": float(df[col].min()),
                    "max": float(df[col].max()),
                    "default": float(df[col].median()),
                    "step": 1.0 if col == "Tenure Months" else 0.1,
                }
                if col in num_cols
                else {
                    "options": sorted(df[col].dropna().astype(str).unique().tolist()),
                    "default": sorted(df[col].dropna().astype(str).unique().tolist())[0],
                }
            ),
        }
        for col in feature_cols
    ],
    "scoring": scoring,
    "cv_mean": float(scores.mean()),
    "cv_std": float(scores.std()),
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Target column: Churn Value")
print(f"Metric ({scoring}): {scores.mean():.4f}")
print(f"Saved model to {model_path}")

Target column: Churn Value
Metric (accuracy): 1.0000
